# 최신 버전 기준 MMR 검색 예제

이 노트북은 기존 `MMR 검색.ipynb`를 **최신 LangChain / OpenAI / Chroma 사용 방식**에 맞게 정리한 별도 파일입니다.

## 반영한 변경 사항

- `from langchain.vectorstores import Chroma` → `from langchain_chroma import Chroma`
- `from langchain.embeddings.openai import OpenAIEmbeddings` → `from langchain_openai import OpenAIEmbeddings`
- `text-embedding-ada-002` → `text-embedding-3-small`
- 하드코딩 API 키 제거, `OPENAI_API_KEY` 환경변수 사용
- 기본 MMR 검색뿐 아니라 **Similarity Search vs MMR Search 비교** 예제 추가
- 최신 retriever 사용 방식(`as_retriever(search_type="mmr")`)도 함께 예시 제공

MMR(Maximal Marginal Relevance)은 **질문과의 관련성**뿐 아니라 **결과들 간의 중복 감소(다양성)**도 함께 고려하는 검색 방식입니다.


In [ ]:
# 필요 패키지 설치
%pip install -qU langchain langchain-openai langchain-chroma python-dotenv


In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

print("OPENAI_API_KEY 준비 완료")


## 1) 예제 텍스트 준비

원본 노트북의 예제 문장을 그대로 활용하되, 최신 Chroma 벡터스토어에 넣어서 검색합니다.


In [ ]:
texts = [
    """ChatGPT 열풍으로 인해 생성형 AI에 대한 관심이 뜨겁다. 생성형 AI는 이용자의 특정 요구에 따라 결과를 능동적으로 생성해 내는 인공지능 기술이다.""",
    """특히, 생성형 AI는 대량의 데이터(Hyper-scale Data)를 학습하여 인간의 영역이라고 할 수 있는 창작의 영역까지 넘보고 있다.""",
    """베타 버전 출시 2개월 만에 MAU(월간 활성 이용자 수)가 무려 1억 명을 넘어섰다. 또한 구글, 메타 등 글로벌 빅테크 기업들이 앞다투어 천문학적인 규모의 투자와 유사 서비스 출시 계획을 발표하고 있다.""",
    """이 서비스의 핵심은 서비스 이용자의 질문을 이해하고 분석하여 수많은 정보 중 답이 될 만한 필요 정보를 스스로 찾아서 이를 적절히 요약과 정리해 제공하는 것이다.""",
    """특히 앞서 질문한 내용의 맥락을 잇거나 구체적인 사례를 들어 질문할수록 더 정확한 답을 얻을 수 있는데, 이는 마치 사람과 대화하는 것처럼 맥락을 이해하여 답을 제공한다는 점에서 이전과 차원이 다른 정보 검색 서비스를 체감하게 한다."""
]

for i, t in enumerate(texts, start=1):
    print(f"[{i}] {t}")


## 2) 최신 방식으로 임베딩 + Chroma 생성

최신 LangChain 문서 기준으로 Chroma는 `langchain-chroma` 패키지에서 가져오고, OpenAI 임베딩은 `langchain-openai` 패키지에서 사용합니다. [LangChain Chroma docs](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) [LangChain OpenAI embeddings docs](https://docs.langchain.com/oss/python/integrations/embeddings/openai)


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = Chroma.from_texts(
    texts=texts,
    embedding=embedding,
    collection_name="mmr_demo_collection",
)

print(vector_store)


## 3) 기본 MMR 검색

원본 노트북의 핵심이었던 `max_marginal_relevance_search(...)`를 최신 방식으로 그대로 살린 예제입니다.

- `k`: 최종 반환 문서 수
- `fetch_k`: 후보로 먼저 가져올 문서 수
- `lambda_mult`: 0에 가까울수록 다양성 강화, 1에 가까울수록 질의 유사성 비중 증가


In [ ]:
question = "생성형 AI에 대해 언급된 내용은?"
mmr_docs = vector_store.max_marginal_relevance_search(
    question,
    k=2,
    fetch_k=4,
    lambda_mult=0.5,
)

for i, doc in enumerate(mmr_docs, start=1):
    print(f"\n[MMR result {i}]\n{doc.page_content}")


## 4) Similarity Search와 MMR Search 비교

일반 similarity search는 **가장 비슷한 문서**를 우선 뽑기 때문에 결과가 서로 비슷비슷할 수 있습니다.
반면 MMR은 비슷한 문서만 몰리지 않도록 **중복을 줄이면서** 뽑는 데 유리합니다.


In [ ]:
question = "생성형 AI 핵심 기능은?"

docs_similarity = vector_store.similarity_search(question, k=2)
docs_mmr = vector_store.max_marginal_relevance_search(question, k=2, fetch_k=5)

print("===== Similarity Search =====")
for i, doc in enumerate(docs_similarity, start=1):
    print(f"\n[similarity {i}]\n{doc.page_content}")

print("\n===== MMR Search =====")
for i, doc in enumerate(docs_mmr, start=1):
    print(f"\n[mmr {i}]\n{doc.page_content}")


## 5) `lambda_mult` 값에 따른 차이 보기

`lambda_mult`는 관련성과 다양성 사이의 균형을 조절합니다.

- `0.1`에 가까울수록: **더 다양한 결과**
- `0.9`에 가까울수록: **질문과 더 유사한 결과 중심**


In [ ]:
question = "생성형 AI 서비스 특징은?"

for lambda_mult in [0.1, 0.5, 0.9]:
    docs = vector_store.max_marginal_relevance_search(
        question,
        k=2,
        fetch_k=5,
        lambda_mult=lambda_mult,
    )
    print(f"\n===== lambda_mult = {lambda_mult} =====")
    for i, doc in enumerate(docs, start=1):
        print(f"\n[result {i}] {doc.page_content}")


## 6) 최신 Retriever 방식으로 MMR 사용

최신 LangChain에서는 벡터스토어를 retriever로 바꿔서 사용하는 패턴도 많이 씁니다.
`search_type="mmr"` 와 `search_kwargs`로 설정할 수 있습니다. [LangChain VectorStore retriever reference](https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/as_retriever)


In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 5, "lambda_mult": 0.5},
)

retrieved_docs = retriever.invoke("생성형 AI가 제공하는 사용자 가치가 무엇인가?")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n[retriever-mmr {i}]\n{doc.page_content}")


## 정리

이 노트북은 원본 MMR 예제를 최신 구조로 바꾼 독립 실행형 버전입니다.

핵심 요약:
- 최신 Chroma import는 `from langchain_chroma import Chroma`
- 최신 OpenAI 임베딩 import는 `from langchain_openai import OpenAIEmbeddings`
- `max_marginal_relevance_search(...)`는 여전히 유효한 MMR 검색 방식
- 실무에서는 `as_retriever(search_type="mmr")` 패턴도 자주 사용
- `fetch_k`, `k`, `lambda_mult` 조합을 조절하면 검색 다양성을 튜닝 가능

필요하면 다음 단계로 이 노트북도 더 확장할 수 있습니다.
- PDF 기반 MMR 검색 버전
- FAISS 기반 MMR 비교 버전
- RAG 질의응답까지 붙인 버전
